In [ ]:
# %% [markdown]
# # Gig Economy Financial Stability Analysis
# ## Phase 1 & 2: Data Loading, Feature Engineering, and Clustering

# %%
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add the 'src' folder to the path so we can import our custom modules
sys.path.append(os.path.abspath('../src'))

# Import our custom functions
from data_loader import load_and_merge_data
from feature_engineering import engineer_stability_features
from clustering import identify_archetypes

# %% [markdown]
# ### 1. Load and Merge Data

# %%
df_raw = load_and_merge_data('../data/raw')

if df_raw is not None:
    print("Data loaded successfully!")
    display(df_raw.head())
else:
    print("Failed to load data. Check your file paths.")

# %% [markdown]
# ### 2. Engineer Features (Create State Variables)

# %%
if df_raw is not None:
    df_eng = engineer_stability_features(df_raw)
    print("\nFeatures engineered. Preview:")
    # Updated column names to match the robust feature engineering code
    display(df_eng[['worker_id', 'Savings_Rate', 'Debt_Pressure']].head())

# %% [markdown]
# ### 3. Identify Financial Archetypes (Clustering)

# %%
if df_raw is not None:
    # Define the features we want to use for clustering
    # Using Savings_Rate (Stability) and Debt_Pressure (Burden)
    features = ['Savings_Rate', 'Debt_Pressure', 'monthly_income', 'Std_Monthly_Income']
    
    # Run the clustering function
    df_clustered, model, scaler = identify_archetypes(df_eng, features=features)
    
    # Show the summary of each cluster
    print("\nCluster Summary (Mean values for each Archetype):")
    display(df_clustered.groupby('Archetype')[features].mean())

# %% [markdown]
# ### 4. Visualize the Archetypes

# %%
if df_raw is not None:
    plt.figure(figsize=(10, 7))
    scatter = plt.scatter(df_clustered['Financial_Stability_Score'], 
                          df_clustered['Debt_Pressure'], 
                          c=df_clustered['Archetype'], 
                          cmap='viridis', alpha=0.6, edgecolors='k')
    
    plt.title('Financial Archetypes of Gig Workers')
    plt.xlabel('Financial Stability Score (Net Cash Flow / Volatility)')
    plt.ylabel('Debt Pressure (Debt-to-Income Ratio)')
    plt.colorbar(scatter, label='Archetype Cluster')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Save the plot
    plt.savefig('../results/figures/archetypes.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Plot saved to results/figures/archetypes.png")

Loading data...
Loaded 120000 workers and 199610 transactions.
Using column 'merchant_category' for transaction types.
Data merging complete.
Data loaded successfully!


,worker_id,timestamp,month_idx,age,occupation,city_tier,annual_income,monthly_income,outstanding_debt,credit_utilization_ratio,...,monthly_balance,credit_score_raw,missed_payments_streak,debt_growth_rate,credit_risk,Avg_Monthly_Income,Std_Monthly_Income,Avg_Monthly_Expense,Avg_Net_Cash_Flow,Num_Months_Active
0,W00000,2022-01-19,0,41,Delivery,1,52617.80,4384.82,75013.88,0.7524,...,2108.00,457.9,2,0.503,High,12.818500,28.362351,139.529500,-126.711000,20
1,W00001,2022-01-25,0,27,Delivery,1,16304.91,1358.74,9836.35,0.2192,...,574.94,677.6,2,0.321,Low,9.284444,32.603936,198.239444,-188.955000,18
2,W00002,2022-01-19,0,36,Driver,1,19334.05,1611.17,32301.13,0.7354,...,226.26,378.0,3,0.503,High,5.758947,13.181195,152.008421,-146.249474,19
3,W00003,2022-01-06,0,22,Freelancer,1,8059.45,671.62,5179.67,0.3769,...,143.18,712.7,0,-0.500,Low,40.139545,74.142844,162.107727,-121.968182,22
4,W00004,2022-01-24,0,29,Delivery,3,45953.23,3829.44,28160.74,0.3357,...,2991.13,698.1,1,0.197,Medium,16.209524,62.809461,183.008095,-166.798571,21


Engineering stability features...
Feature engineering complete. 120000 workers in dataset.

Features engineered. Preview:


KeyError: "['Financial_Stability_Score'] not in index"

In [ ]:
# %% [markdown]
# ### 5. Robust Stability Analysis (PhD-Level)

# %%
import importlib
import stability_analysis
importlib.reload(stability_analysis)

from stability_analysis import estimate_cluster_stability_robust

# Run the robust analysis
stability_results_robust = estimate_cluster_stability_robust(df_clustered)

# Display Results
print("\n--- Robust Stability Analysis Results ---")
for cluster_id, metrics in stability_results_robust.items():
    print(f"\nArchetype {cluster_id}:")
    print(f"  Debt Sensitivity: {metrics['Debt_Sensitivity']:.4f}")
    print(f"  Baseline Savings Rate: {metrics['Baseline_Savings_Rate']:.4f}")
    print(f"  Status: {metrics['Status']}")
    print(f"  Sample Size: {metrics['Sample_Size']}")

Performing Robust Stability Analysis...

--- Robust Stability Analysis Results ---

Archetype 0:
  Debt Sensitivity: -0.0192
  Baseline Savings Rate: -0.6090
  Status: Moderately Sensitive
  Sample Size: 96


In [ ]:
# Check how many workers have valid Income, Debt, and Net Cash Flow
print("Total Workers:", len(df_clustered))

# Check for NaNs in key columns
print("\nNaN counts:")
print("Avg_Monthly_Income:", df_clustered['Avg_Monthly_Income'].isna().sum())
print("outstanding_debt:", df_clustered['outstanding_debt'].isna().sum())
print("Avg_Net_Cash_Flow:", df_clustered['Avg_Net_Cash_Flow'].isna().sum())

# Check how many pass the income filter
df_valid_income = df_clustered[df_clustered['Avg_Monthly_Income'] > 100]
print("\nWorkers with Income > 100:", len(df_valid_income))

# Check if Debt_Pressure was calculated correctly
if 'Debt_Pressure' in df_clustered.columns:
    print("Debt_Pressure NaNs:", df_clustered['Debt_Pressure'].isna().sum())
    print("Debt_Pressure Sample:", df_clustered['Debt_Pressure'].head())
else:
    print("ERROR: Debt_Pressure column missing!")

Total Workers: 120000

NaN counts:
Avg_Monthly_Income: 0
outstanding_debt: 0
Avg_Net_Cash_Flow: 0

Workers with Income > 100: 96
Debt_Pressure NaNs: 0
Debt_Pressure Sample: 0    17.107630
1     7.239317
2    20.048244
3     7.712203
4     7.353749
Name: Debt_Pressure, dtype: float64
